In [ ]:
# rag_plus_mcp_runner.py
# RAG murni -> sanitizer identifier (backtick FQN) -> anti-JOIN (alias & no-alias)
# -> DISTINCT heuristik cerdas -> eksekusi via MCP (+ optional export)
# FIXED: Consistent minimal-JOIN SQL generation

import os
import re
import json
import glob
import hashlib
import asyncio
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

from langgraph.graph import StateGraph, END
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Optional (Jupyter)
try:
    from IPython.display import HTML, display
except Exception:
    def display(*_a, **_k): pass
    class HTML:  # type: ignore
        def __init__(self, *_a, **_k): pass

# MCP client
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# Pretty table
import pandas as pd
from tabulate import tabulate

# =========================
# === CONFIG (ENV) ========
# =========================
# ---- RAG ----
JSON_INPUT = os.getenv("EXPORT_JSONL_OUTPUT", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR = os.getenv("CHROMA_DIR", "./chroma_db")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "classicmodels")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen2.5:3b")
TOP_K = int(os.getenv("TOP_K", "5"))

RETRIEVAL_PROBE = (
    " measure measures metric metrics sum total amount price revenue sales gross net "
    "quantity qty count average avg mean distinct group by bucket date time timestamp "
    "created updated order_date month week quarter year category region customer status id"
)

# ---- MCP ----
MCP_BASE    = os.getenv("MCP_BASE", "http://mcp_server:8000/mcp")
MCP_TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", "60"))

# ---- Export ----
EXPORT_DIR = os.getenv("EXPORT_DIR", "exports")
JUPYTER_BASE_URL = os.getenv("JUPYTER_BASE_URL", "").rstrip("/")

# =========================
# === RAG (PERSIS AWAL) ===
# =========================
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str  # SQL (+ optional notes)

def resolve_jsonl_paths(spec: str) -> List[str]:
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return out

def build_vector_store() -> Chroma:
    embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    return Chroma(
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
    )

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

def _safe_iso_to_dt(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None

def _stable_id(meta: Dict[str, Any], source_name: str) -> str:
    key = f"{meta.get('database','')}|{meta.get('table','')}|{meta.get('column','')}|{source_name}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()

def show_graph_mermaid_only(app) -> None:
    try:
        G = app.get_graph()
        mermaid_src = G.draw_mermaid()
    except Exception:
        return
    html = f"""
<div class="mermaid">
{mermaid_src}
</div>
<script>
(function () {{
  function renderMermaid() {{
    if (window.mermaid) {{
      mermaid.initialize({{ startOnLoad: true, securityLevel: 'loose' }});
      mermaid.init();
    }}
  }}
  if (!window.mermaid) {{
    var s = document.createElement('script');
    s.src = 'https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js';
    s.onload = renderMermaid;
    document.body.appendChild(s);
  }} else {{
    renderMermaid();
  }}
}})();
</script>
"""
    display(HTML(html))

def ingest_node(state: RAGState) -> RAGState:
    paths = state["jsonl_paths"]
    vs = build_vector_store()

    total_in_collection = vs._collection.count()
    first_run = (total_in_collection == 0)
    if first_run:
        print("First run detected (empty collection) - embedding all documents.")

    added = 0
    updated = 0
    skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"JSONL not found: {path}")
            continue

        docs: List[Document] = []
        ids: List[str] = []
        metas: List[Dict[str, Any]] = []

        for raw, obj in read_jsonl(path):
            meta: Dict[str, Any] = obj if isinstance(obj, dict) else {"raw": raw}
            meta["_source"] = p.name
            metas.append(meta)
            ids.append(_stable_id(meta, p.name))
            docs.append(Document(page_content=raw, metadata=meta))

        if not docs:
            continue

        if first_run:
            vs.add_documents(docs, ids=ids)
            added += len(docs)
            print(f"Embedded {len(docs)} docs from {path}")
            continue

        existing_meta_by_id: Dict[str, Dict[str, Any]] = {}
        BATCH = 1000
        for i in range(0, len(ids), BATCH):
            chunk_ids = ids[i:i + BATCH]
            got = vs._collection.get(ids=chunk_ids, include=["metadatas"])
            got_ids = got.get("ids") or []
            got_metas = got.get("metadatas") or []
            for doc_id, md in zip(got_ids, got_metas):
                existing_meta_by_id[doc_id] = md or {}

        to_upsert_docs: List[Document] = []
        to_upsert_ids: List[str] = []

        for doc, doc_id, meta in zip(docs, ids, metas):
            incoming_li = meta.get("last_ingested")
            existing = existing_meta_by_id.get(doc_id)

            if not existing:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
                continue

            stored_li = existing.get("last_ingested")
            should_update = (incoming_li != stored_li)

            in_dt = _safe_iso_to_dt(incoming_li)
            st_dt = _safe_iso_to_dt(stored_li)
            if in_dt and st_dt:
                should_update = in_dt > st_dt

            if should_update:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
            else:
                skipped += 1

        if to_upsert_docs:
            vs.delete(ids=to_upsert_ids)
            vs.add_documents(to_upsert_docs, ids=to_upsert_ids)
            updated += len(to_upsert_docs)
            print(f"Upserted {len(to_upsert_docs)} changed docs from {path}")

    try:
        vs.persist()
    except Exception:
        pass

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    return state

def retrieve_node(state: RAGState) -> RAGState:
    vs = build_vector_store()
    retriever = vs.as_retriever(search_kwargs={"k": max(3, TOP_K)})
    q = (state["question"] or "").strip()
    hybrid_query = (q + " " + RETRIEVAL_PROBE).strip()
    docs = retriever.invoke(hybrid_query)
    state["context"] = "\n".join(d.page_content for d in (docs or []))
    return state

# =========================
# === Sanitizer & SQL PP ==
# =========================
_AGG_WORDS = re.compile(r"\b(sum|avg|count|min|max)\s*\(", re.I)
JOIN_KW = r"(?:inner\s+join|left\s+join|right\s+join|full\s+join|join)"

def _normalize_cols(s: str) -> List[str]:
    parts = [p.strip() for p in s.split(",")]
    out = []
    for p in parts:
        m = re.split(r"\s+AS\s+", p, flags=re.I)
        left = m[0]
        out.append(left.strip())
    return out

def _has_join(sql: str) -> bool:
    return re.search(rf"\b{JOIN_KW}\b", sql, flags=re.I) is not None

def _prefer_distinct_over_group_by(sql: str) -> str:
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+(.*?)(\s+group\s+by\s+(.*?))(?:\s+order\s+by\s+.*)?;", sql, flags=re.I | re.S)
    if not m:
        return sql
    select_cols = _normalize_cols(m.group(1))
    group_by_cols = _normalize_cols(m.group(4))
    if _AGG_WORDS.search(sql):
        return sql
    if select_cols and group_by_cols and set(map(str.lower, select_cols)) == set(map(str.lower, group_by_cols)):
        sql2 = re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)
        sql2 = re.sub(r"\s+group\s+by\s+.*?(?=(\s+order\s+by\s+.*)?;)", "", sql2, flags=re.I | re.S)
        return sql2
    return sql

def _split_qualified(name: str) -> List[str]:
    """Split a dotted identifier (schema.table[.col]) into parts (without backticks)."""
    return [p.strip('`" ') for p in name.split(".") if p.strip("`\" ")]

def _short_table(name_or_fqn: str) -> str:
    """Return short table name from schema.table or table."""
    parts = _split_qualified(name_or_fqn)
    return parts[-1] if parts else name_or_fqn.strip('`"')

def _fix_bad_backticks(sql: str) -> str:
    """
    Perbaiki kasus: `classicmodels.customers` atau `db.tbl.col` yang dibungkus 1 backtick.
    Ubah menjadi `classicmodels`.`customers`[.`col`], atau jika bukan identifier,
    lepas backtick-nya.
    """
    def repl_fqn(m: re.Match) -> str:
        inside = m.group(1)
        if "." not in inside:
            return f"`{inside}`"
        parts = _split_qualified(inside)
        if len(parts) == 0:
            return inside
        return ".".join(f"`{p}`" for p in parts)

    # 1) Backtick block yang berisi titik -> pecah jadi part-part
    sql = re.sub(r"`([^`]+?\.[^`]+?)`", repl_fqn, sql)

    # 2) a.`b.c`  -> a.`b`.`c`
    sql = re.sub(r"(\b[A-Za-z_][\w]*)\.`([^`]+?)`", lambda m:
                 f"{m.group(1)}." + ".".join(f"`{p}`" for p in _split_qualified(m.group(2))), sql)

    #    `a.b`.c  -> `a`.`b`.c
    sql = re.sub(r"`([^`]+?)`\.(\b[A-Za-z_][\w]*)", lambda m:
                 ".".join(f"`{p}`" for p in _split_qualified(m.group(1))) + f".{m.group(2)}", sql)

    return sql

def _select_qualifiers(sql: str) -> set:
    """
    Return set of qualifiers used in the SELECT list (alias or short table).
    """
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+", sql, flags=re.I | re.S)
    if not m:
        return set()
    select = m.group(1)
    quals = set()

    for q in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\.", select):
        quals.add(q.strip('`"'))

    for sch, tbl, col in re.findall(r"`([^`]+)`\s*\.\s*`([^`]+)`\s*\.\s*`([^`]+)`", select):
        quals.add(tbl)

    for tbl, col in re.findall(r"`([^`]+)`\s*\.\s*`([^`]+)`", select):
        quals.add(tbl)

    return quals

def _has_filters(sql: str) -> bool:
    """True if the query contains WHERE / GROUP BY / HAVING."""
    return bool(re.search(r"\b(where|group\s+by|having)\b", sql, flags=re.I))

def _strip_all_joins_unconditionally(sql: str) -> str:
    """Remove every ' JOIN ... ON ... ' block (INNER/LEFT/RIGHT/FULL), keep clauses."""
    return re.sub(
        rf"\s+{JOIN_KW}\s+[`\"\w\.]+(?:\s+[`\"\w]+)?\s+on\s+[^;]+?(?=\b({JOIN_KW}|where|group|order|limit|having|;)\b)",
        "",
        sql,
        flags=re.I | re.S
    )

def _find_anchor_qualifier(sql: str) -> Optional[str]:
    """Return alias (if any) else short table name from FROM."""
    m = re.search(r"\bfrom\s+([`\"\w\.]+)(?:\s+([`\"\w]+))?", sql, flags=re.I)
    if not m:
        return None
    table_fqn = m.group(1)
    alias = m.group(2)
    return (alias or _short_table(table_fqn)).strip('`"')

def _remove_redundant_joins(sql: str) -> str:
    """
    Hapus JOIN jika qualifier-nya tidak dipakai di luar ON.
    Support JOIN tanpa alias.
    """
    pattern = re.compile(
        rf"\b({JOIN_KW})\s+([`\"\w\.]+)(?:\s+([`\"\w]+))?\s+on\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)",
        re.I | re.S
    )
    joins = list(pattern.finditer(sql))
    if not joins:
        return sql

    def alias_used_outside_on(qualifier: str, sql_text: str, on_span: tuple[int,int], table_name_short: str) -> bool:
        before = sql_text[:on_span[0]]
        after  = sql_text[on_span[1]:]
        pat1 = re.compile(rf"\b{re.escape(qualifier)}\s*\.", re.I)
        pat2 = re.compile(rf"\b{re.escape(table_name_short)}\s*\.", re.I)
        return bool(pat1.search(before) or pat1.search(after) or pat2.search(before) or pat2.search(after))

    new_sql = sql
    offset = 0
    for j in joins:
        table_fqn = j.group(2).strip('`"')
        alias = j.group(3).strip('`"') if j.group(3) else None
        qualifier = alias or _short_table(table_fqn)
        on_start = j.start(4) + offset
        on_end   = j.end(4) + offset
        block_span = (j.start() + offset, j.end() + offset)
        used = alias_used_outside_on(qualifier, new_sql, (on_start, on_end), _short_table(table_fqn))
        if not used:
            new_sql = new_sql[:block_span[0]] + new_sql[block_span[1]:]
            offset -= (block_span[1] - block_span[0])
    return new_sql

def _strip_on_clauses(sql: str) -> str:
    return re.sub(rf"\bon\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)", " ON ", sql, flags=re.I | re.S)

def _qualifiers_used_outside_on(sql: str) -> set:
    stripped = _strip_on_clauses(sql)
    quals = set(m.group(1).strip('`"') for m in re.finditer(r"\b([`\"\w]+)\s*\.", stripped))
    return set(q.strip('`"') for q in quals if q)

def _pushdown_filters_through_equi_join(sql: str) -> str:
    """
    ON a.x = b.y -> ganti referensi b.y di luar ON menjadi a.x bila anchor = a.
    """
    anchor = _find_anchor_qualifier(sql)
    if not anchor:
        return sql

    pattern = re.compile(
        rf"\b({JOIN_KW})\s+([`\"\w\.]+)(?:\s+([`\"\w]+))?\s+on\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)",
        re.I | re.S
    )
    alias_map: Dict[str, str] = {}
    joins = list(pattern.finditer(sql))
    for j in joins:
        table_fqn = j.group(2).strip('`"')
        alias = j.group(3).strip('`"') if j.group(3) else None
        qualifier = alias or _short_table(table_fqn)
        on_text = j.group(4)

        for m in re.finditer(r"\b([`\"\w]+)\.([`\"\w]+)\s*=\s*([`\"\w]+)\.([`\"\w]+)\b", on_text):
            a1, c1, a2, c2 = (m.group(1).strip('`"'), m.group(2).strip('`"'),
                               m.group(3).strip('`"'), m.group(4).strip('`"'))
            if a1 == anchor and a2 in {qualifier, _short_table(table_fqn)}:
                alias_map[f"{qualifier}.{c2}"] = f"{anchor}.{c1}"
            elif a2 == anchor and a1 in {qualifier, _short_table(table_fqn)}:
                alias_map[f"{qualifier}.{c1}"] = f"{anchor}.{c2}"

    if not alias_map:
        return sql

    new_sql = sql
    for j in joins:
        on_span = (j.start(4), j.end(4))
        before = new_sql[:on_span[0]]
        onpart = new_sql[on_span[0]:on_span[1]]
        after  = new_sql[on_span[1]:]
        for alias_col, anchor_col in alias_map.items():
            qname, col = alias_col.split(".")
            alias_regex = re.compile(rf"\b{re.escape(qname)}\s*\.\s*{re.escape(col)}\b", re.I)
            before = alias_regex.sub(anchor_col, before)
            after  = alias_regex.sub(anchor_col, after)
        new_sql = before + onpart + after

    new_sql = _remove_redundant_joins(new_sql)
    return new_sql

_IDLIKE = re.compile(r".*\b(id|.*_id|.*id$|.*number$|.*_number)\b", re.I)

def _select_has_anchor_id_like(sql: str, anchor: Optional[str]) -> bool:
    if not anchor:
        return False
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+", sql, flags=re.I | re.S)
    if not m:
        return False
    cols = _normalize_cols(m.group(1))
    for col in cols:
        m2 = re.match(r"(?:(?:`?)([\w]+)(?:`?)\s*\.\s*(?:`?)([\w]+)(?:`?))", col)
        if not m2:
            continue
        qualifier, cname = m2.group(1), m2.group(2)
        if qualifier.strip('`"').lower() == anchor.lower() and _IDLIKE.match(cname):
            return True
    return False

def _drop_unnecessary_distinct(sql: str) -> str:
    if "distinct" not in sql.lower():
        return sql
    if _has_join(sql) or " group by " in sql.lower() or _AGG_WORDS.search(sql):
        return sql
    anchor = _find_anchor_qualifier(sql)
    if _select_has_anchor_id_like(sql, anchor):
        return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)
    return sql

def _strip_where_non_anchor_refs(sql: str) -> str:
    """
    Buang predicate WHERE yang merujuk qualifier non-anchor (setelah join dibuang).
    """
    anchor = _find_anchor_qualifier(sql)
    if not anchor:
        return sql
    m = re.search(r"\bwhere\b(.*?)(?=\border\b|\bgroup\b|\blimit\b|;|$)", sql, flags=re.I | re.S)
    if not m:
        return sql
    where_body = m.group(1)
    start, end = m.span(1)

    parts = re.split(r"(\band\b|\bor\b)", where_body, flags=re.I)
    kept_parts = []
    for p in parts:
        if re.fullmatch(r"\s*(and|or)\s*", p, flags=re.I):
            kept_parts.append(p)
            continue
        qualifiers = re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\.", p)
        qualifiers = [q for q in qualifiers if q.lower() != anchor.lower()]
        if qualifiers:
            continue
        kept_parts.append(p)

    new_where_body = "".join(kept_parts).strip()
    if not new_where_body:
        sql = sql[:m.start()] + sql[m.end():]
    else:
        sql = sql[:start] + new_where_body + sql[end:]
    return sql

def _maybe_add_limit(question: str, sql: str) -> str:
    if re.search(r"\blimit\s+\d+\b", sql, flags=re.I):
        return sql
    m = re.search(r"\b(\d{1,4})\b", question)
    if m:
        n = int(m.group(1))
        n = max(1, min(n, 1000))
        s = sql.rstrip().rstrip(";")
        return s + f" LIMIT {n};"
    return sql

# ====== DISTINCT heuristik cerdas ======

def _wants_distinct_from_question(question: str) -> bool:
    q = (question or "").lower()
    return any(k in q for k in [
        "distinct", "unique", "no duplicate", "tanpa duplikat",
        "unik", "dedup", "de-dup", "de dup", "hanya yang unik"
    ])

def _wants_all_rows_from_question(question: str) -> bool:
    q = (question or "").lower()
    return any(k in q for k in [
        "all rows", "include duplicates", "dengan duplikat",
        "raw rows", "semua baris", "tanpa dedup", "jangan distinct"
    ])

def _is_grouping_or_agg(sql: str) -> bool:
    s = sql.lower()
    return (" group by " in s) or (" having " in s) or bool(_AGG_WORDS.search(sql))

def _has_distinct(sql: str) -> bool:
    return bool(re.search(r"^\s*select\s+distinct\b", sql, flags=re.I))

def _add_distinct(sql: str) -> str:
    if _has_distinct(sql):
        return sql
    return re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)

def _remove_distinct(sql: str) -> str:
    return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)

def _ensure_reasonable_distinct(sql: str, question: str) -> str:
    """
    Kebijakan DISTINCT yang 'masuk akal':
    - Jika user minta unique/distinct -> pakai DISTINCT.
    - Jika user minta semua baris/duplikat -> hapus DISTINCT.
    - Jika tanpa JOIN dan tanpa GROUP/HAVING/AGG:
        * Bila SELECT mengandung kolom id-like anchor -> hapus DISTINCT.
        * Selain itu -> tambahkan DISTINCT (mis. nama customer).
    - Jika ada JOIN dan tidak ada GROUP/HAVING/AGG:
        * Asumsikan potensi duplikasi -> tambahkan DISTINCT (kecuali user minta all rows).
    """
    if not sql:
        return sql

    if _wants_all_rows_from_question(question):
        return _remove_distinct(sql)
    if _wants_distinct_from_question(question):
        return _add_distinct(sql)

    has_join = _has_join(sql)
    is_group_or_agg = _is_grouping_or_agg(sql)
    anchor = _find_anchor_qualifier(sql)

    if is_group_or_agg:
        return sql

    if not has_join:
        if _select_has_anchor_id_like(sql, anchor):
            return _remove_distinct(sql)
        else:
            return _add_distinct(sql)
    else:
        return _add_distinct(sql)

# ====== Orkestrasi Postprocess ======

def _postprocess_sql(sql: str, question: str) -> str:
    if not sql:
        return sql
    s = sql.strip()
    if not s.endswith(";"):
        s += ";"

    # 1) Fix backtick FQN issues first
    s = _fix_bad_backticks(s)

    # 2) Anti-JOIN transforms (alias & no-alias)
    s = _pushdown_filters_through_equi_join(s)
    s = _remove_redundant_joins(s)
    s = _prefer_distinct_over_group_by(s)
    s = _drop_unnecessary_distinct(s)
    s = _strip_where_non_anchor_refs(s)

    # 2.5) ENHANCED HARD GUARD:
    # If SELECT uses only the anchor qualifier AND there are no filters/grouping/having,
    # then strip ALL JOIN blocks outright.
    anchor = _find_anchor_qualifier(s)
    select_quals = _select_qualifiers(s)
    
    if anchor and select_quals:
        # Check if ONLY the anchor qualifier is used in SELECT
        non_anchor_quals = select_quals - {anchor}
        has_filtering = _has_filters(s)
        
        # If SELECT only uses anchor and there's no filtering, nuke all JOINs
        if not non_anchor_quals and not has_filtering:
            if _has_join(s):
                print(f"Removing unnecessary JOINs (only {anchor} columns selected, no filtering)")
                s = _strip_all_joins_unconditionally(s)

    # 2.8) DISTINCT yang cerdas (berbasis intent + struktur SQL)
    s = _ensure_reasonable_distinct(s, question)

    # 3) Auto LIMIT jika user minta N item
    s = _maybe_add_limit(question, s)
    return s

# ---------- Prompt ----------
def _make_generation_prompt(ctx: str, question: str) -> str:
    return f"""
You are a senior SQL assistant for MySQL 8.0.

Return a runnable query using ONLY columns/tables in Context.

########################
# STRICT OUTPUT FORMAT #
########################
{{
  "sql": "<final MySQL query>",
  "notes": "<brief rationale>"
}}
No extra keys. No Markdown. End SQL with a semicolon.

##################
# SQL CONSTRAINT #
##################
- MySQL 8.0; backticks only when needed; ANSI joins.

########################################
# CRITICAL: MINIMIZE JOINS - PRIORITY #1
########################################
**DEFAULT RULE**: Use ONLY the anchor table unless absolutely necessary.

**Before adding ANY JOIN, ask yourself:**
1. "Does the question explicitly require data from this joined table?"
2. "Is there a column in the anchor table that already provides this information?"

**When in doubt, prefer the simpler query WITHOUT JOINs.**

########################################
# ANCHOR TABLE & MINIMAL-JOIN POLICY   #
########################################
- Pick as ANCHOR the table containing the most required columns.
- If the ANCHOR already has a needed field by name/description, USE IT. Do NOT JOIN another table for a duplicate.
- Only JOIN if you SELECT, GROUP BY, or WHERE on a column that is NOT in the ANCHOR.
- If an equi-join ties ANCHOR.col = DIM.col and WHERE only constrains DIM.col with a literal, PUSH the filter to ANCHOR.col and REMOVE the JOIN unless DIM also contributes a selected/grouped column.
- After drafting, AUDIT each JOIN: if you cannot state which output/filter column comes from that table, REMOVE it.

########################################
# NOTES MUST MATCH THE FINAL SQL
########################################
- Do NOT mention tables/operations that are NOT present in the final SQL.
- For simple queries (no joins), just say "Simple query on [table_name] table."
- For queries with joins, explain which specific column from the joined table is being used in SELECT/WHERE/GROUP BY.

########################################
# AGGREGATION & DISTINCT
########################################
- Use SUM/AVG/COUNT as appropriate; COUNT(DISTINCT x) for unique entities.
- Avoid DISTINCT unless: (a) the user explicitly asks for deduplication, or (b) you have a JOIN that creates duplicates and you must dedupe output columns.
- If there is no aggregation and only dedup by dimensions was intended, prefer SELECT DISTINCT over GROUP BY; otherwise avoid both.

########################################
# DATE BUCKETING (MySQL-safe)
########################################
- DATE(dt) AS day; DATE_FORMAT(dt,'%Y-%m') AS month; CONCAT(YEAR(dt),'-Q',QUARTER(dt)) AS quarter; YEAR(dt) AS year.
- Always GROUP BY any selected bucket/dimension.

#################################
# FILTERS & TYPES
#################################
- Do not invent date ranges. Cast textual dates with STR_TO_DATE if needed.
- Only add WHERE conditions clearly implied by the Question.

################
# REAL INPUTS  #
################
Context:
{ctx}

Question:
{question}
""".strip()

def generate_node(state: RAGState) -> RAGState:
    # FIXED: Force deterministic generation
    llm = ChatOllama(
        model=CHAT_MODEL, 
        base_url=OLLAMA_BASE_URL, 
        temperature=0.0,  # Changed from 0.2 to 0.0 for consistency
        seed=42  # Added seed for reproducibility
    )
    prompt = _make_generation_prompt(state.get("context", ""), state.get("question", ""))
    resp = llm.invoke(prompt)
    content = getattr(resp, "content", None) or str(resp)

    answer_out = content
    try:
        obj = json.loads(content)
        raw_sql = (obj.get("sql") or "").strip()
        notes = (obj.get("notes") or "").strip()

        # Post-process SQL (backtick fix, anti-JOIN, DISTINCT cerdas, dsb.)
        sql = _postprocess_sql(raw_sql, state.get("question","")) if raw_sql else raw_sql

        # ADDED: Post-generation validation for simple queries
        question_lower = state.get("question", "").lower()
        simple_keywords = ["show", "list", "get", "what are", "display", "give me"]
        complex_keywords = ["who ordered", "with orders", "placed", "purchased", "bought", "having"]
        
        is_simple_query = any(k in question_lower for k in simple_keywords)
        is_complex_query = any(k in question_lower for k in complex_keywords)
        
        # If simple query still has JOINs and shouldn't, force removal
        if is_simple_query and not is_complex_query and "join" in sql.lower():
            print("Detected unnecessary JOIN in simple query - forcing removal")
            sql = _strip_all_joins_unconditionally(sql)

        # Samakan notes dgn SQL final
        if sql:
            simple = not re.search(r"\b(join|union|where|group\s+by|having)\b", sql, flags=re.I)
            if simple:
                # Extract table name for simple queries
                m = re.search(r"\bfrom\s+[`\"]?(?:\w+\.)?(\w+)[`\"]?", sql, flags=re.I)
                table_name = m.group(1) if m else "table"
                notes = f"Simple query on {table_name} table."
            else:
                if re.search(r"\b(join|union)\b", notes, flags=re.I) and not re.search(r"\b(join|union)\b", sql, flags=re.I):
                    notes = "Query executed without joins."
        if sql:
            answer_out = sql + ("\n\n-- notes: " + notes if notes else "")
        else:
            answer_out = (notes or "").strip() or content
    except Exception:
        pass

    state["answer"] = answer_out
    return state

def build_app():
    g = StateGraph(RAGState)
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("generate", generate_node)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

# =========================
# === MCP UTIL ============
# =========================
async def _mcp_call(client: Client, tool: str, payload: dict | None = None, timeout: int = MCP_TIMEOUT):
    coro = client.call_tool(tool, payload or {})
    return await asyncio.wait_for(coro, timeout=timeout)

def _extract_payload(result) -> Any:
    if hasattr(result, 'data') and result.data is not None:
        return result.data
    if hasattr(result, "structuredContent") and result.structuredContent is not None:
        return result.structuredContent
    if hasattr(result, "content") and result.content:
        return result.content[0].text if isinstance(result.content, list) else str(result.content)
    return result

def _ensure_json(d: Any) -> Optional[dict]:
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        try:
            return json.loads(d)
        except Exception:
            return None
    return None

def display_tables_list(data: dict):
    tables = data.get("tables", [])
    total = data.get("total_tables", len(tables))
    if not tables:
        print("No tables found.")
        return
    print("\nAvailable tables:")
    for i, t in enumerate(tables, start=1):
        print(f"  {i:>2}. {t}")
    print(f"\nTotal: {total} table(s)")

def display_table_from_json(data):
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))
    if preview_rows and preview_cols:
        df = pd.DataFrame(preview_rows, columns=preview_cols)
        df.index = df.index + 1
        if total_rows > len(preview_rows):
            df.loc["..."] = ["..."] * len(df.columns)
        if total_cols > len(df.columns):
            df["..."] = "..."
        print("\nQuery Result (preview):")
        print(tabulate(df, headers="keys", tablefmt="pretty"))
        print(f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
              f"(full result: {total_rows} rows × {total_cols} columns)")
        dbg = data.get("_debug_info", {}) or {}
        if dbg:
            print(f"Debug: Query returned {dbg.get('query_returned_rows', '?')} rows, "
                  f"True total: {dbg.get('true_total_rows', '?')} rows")
    else:
        print("Query executed but returned no rows.")

def _get_excel_engine() -> str | None:
    try:
        import openpyxl  # noqa
        return "openpyxl"
    except Exception:
        pass
    try:
        import xlsxwriter  # noqa
        return "xlsxwriter"
    except Exception:
        pass
    return None

def _save_full_to_excel(columns: list, rows: list, suggested_name: str | None, output_dir: str) -> str:
    os.makedirs(output_dir, exist_ok=True)
    df = pd.DataFrame(rows, columns=columns)
    stamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    base = suggested_name or "export"
    engine = _get_excel_engine()
    if engine:
        fname = f"export_{base}_{stamp}.xlsx"
        full_path = os.path.join(output_dir, fname)
        with pd.ExcelWriter(full_path, engine=engine) as writer:
            df.to_excel(writer, index=False)
        return os.path.abspath(full_path)
    else:
        fname = f"export_{base}_{stamp}.csv"
        full_path = os.path.join(output_dir, fname)
        df.to_csv(full_path, index=False)
        print("Neither 'openpyxl' nor 'XlsxWriter' is installed. Saved CSV instead.")
        print("   To get .xlsx, install one of:\n   - pip install openpyxl\n   - pip install XlsxWriter")
        return os.path.abspath(full_path)

def _jupyter_files_url(abs_path: str) -> Optional[str]:
    try:
        root = os.getcwd()
        rel = os.path.relpath(abs_path, root)
        if rel.startswith(".."):
            return None
        return "/files/" + rel.replace(os.sep, "/")
    except Exception:
        return None

# ======== ADJUSTMENT START: helper for consistent export prompt ========
async def _maybe_offer_export(mcp: Client, payload: Optional[dict], answer_sql: str):
    # Offer export only if the quick preview succeeded
    if not isinstance(payload, dict) or not payload.get("success"):
        return

    yn = input("\nExport FULL result to Excel/CSV? [y/N]: ").strip().lower()
    if yn not in ("y", "yes"):
        return

    try:
        dbg = payload.get("_debug_info", {}) or {}
        base_query = dbg.get("base_query") or answer_sql.split("\n\n-- notes:")[0].strip().rstrip(";")

        full = await _mcp_call(mcp, "query_db_full", {"sql": base_query})
        full_payload = _ensure_json(_extract_payload(full))

        if isinstance(full_payload, dict) and full_payload.get("success"):
            cols = full_payload.get("columns", [])
            rows = full_payload.get("rows", [])
            os.makedirs(EXPORT_DIR, exist_ok=True)

            m = re.search(r'from\s+("?([A-Za-z0-9_]+)"?\.)?"?([A-Za-z0-9_]+)"?', base_query, flags=re.I)
            suggested = m.group(3) if m else "export"

            outpath = _save_full_to_excel(cols, rows, suggested, EXPORT_DIR)
            print(f"Saved: {outpath}")
            rel = _jupyter_files_url(outpath)
            if rel:
                url = f"{JUPYTER_BASE_URL}{rel}" if JUPYTER_BASE_URL else rel
                print(f"Download via Jupyter: {url}")
            if full_payload.get("truncated"):
                print("Export was truncated to server cap.")
        else:
            print("Failed to export full result.")
    except Exception as e:
        print(f"Export error: {e}")
# ======== ADJUSTMENT END ========

# =========================
# === REPL RUNNER =========
# =========================
async def repl(paths: List[str]):
    app = build_app()
    show_graph_mermaid_only(app)

    transport = StreamableHttpTransport(MCP_BASE)
    async with Client(transport=transport) as mcp:
        try:
            tools = await mcp.list_tools()
            names = [getattr(t, "name", None) for t in tools]
            print("\nMCP tools:", ", ".join([n for n in names if n]) or "(none)")
        except Exception as e:
            print(f"Could not list MCP tools: {e}")

        first_q = input("Enter a test question (or press Enter to skip): ").strip()
        if not first_q:
            first_q = "Which columns describe customer identity?"

        init: RAGState = {
            "jsonl_paths": paths,
            "question": first_q,
            "added": 0,
            "updated": 0,
            "skipped": 0,
            "context": "",
            "answer": "",
        }
        out = app.invoke(init)

        vs = build_vector_store()
        total = vs._collection.count()
        print("\n=== RAG Ingest Report ===")
        print(f"Added (first run): {out.get('added', 0)}")
        print(f"Updated (upserted): {out.get('updated', 0)}")
        print(f"Skipped (unchanged): {out.get('skipped', 0)}")
        print(f"Total docs now in collection: {total}")
        print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")

        print("\n--- Generated Answer (RAG) ---\n")
        generated = out.get("answer", "(no answer)")
        print(generated)

        async def exec_via_mcp(sql_block: str):
            # gunakan SQL sebelum notes; sudah ter-sanitize & postprocess
            sql_only = sql_block.split("\n\n-- notes:")[0].strip()
            sql_clean = sql_only.rstrip(";").strip()

            try:
                lt = await _mcp_call(mcp, "list_tables", {})
                lt_payload = _ensure_json(_extract_payload(lt)) or {}
                if lt_payload.get("success"):
                    display_tables_list(lt_payload)
            except Exception as e:
                print(f"list_tables failed (skip): {e}")

            print(f"\nSQL (RAG -> MCP):\n{sql_clean};")

            try:
                res = await _mcp_call(mcp, "query_db", {"sql": sql_clean})
                payload = _ensure_json(_extract_payload(res))
                if isinstance(payload, dict) and payload.get("success"):
                    display_table_from_json(payload)
                    return payload
                else:
                    if isinstance(payload, dict) and payload.get("suggestions"):
                        print("\nTable not found. Suggestions:")
                        for s in payload["suggestions"]:
                            print(" -", s)
                    if isinstance(payload, dict) and payload.get("column_suggestions"):
                        print("\nColumn suggestions:")
                        for s in payload["column_suggestions"]:
                            print(" -", s)
                    err = payload.get("error") if isinstance(payload, dict) else str(payload)
                    print(f"query_db failed: {err}")
                    return None
            except asyncio.TimeoutError:
                print("MCP query_db timeout")
                return None
            except Exception as e:
                print(f"MCP query_db error: {e}")
                return None

        payload = await exec_via_mcp(generated)
        # ======== ADJUSTMENT START: offer export after initial query ========
        await _maybe_offer_export(mcp, payload, generated)
        # ======== ADJUSTMENT END ========

        print("\nAsk me more questions! (type 'exit' to quit)")
        while True:
            q = input("\nQuestion> ").strip()
            if not q or q.lower() in {"exit", "quit", "q"}:
                print("Bye.")
                break

            state: RAGState = {
                "jsonl_paths": paths,
                "question": q,
                "added": 0,
                "updated": 0,
                "skipped": 0,
                "context": "",
                "answer": "",
            }
            state = retrieve_node(state)
            state = generate_node(state)

            print("\n--- Generated Answer (RAG) ---\n")
            print(state.get("answer", "(no answer)"))

            payload = await exec_via_mcp(state.get("answer", ""))

            # ======== ADJUSTMENT START: use helper for export prompt in REPL ========
            await _maybe_offer_export(mcp, payload, state.get("answer", ""))
            # ======== ADJUSTMENT END ========

def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set EXPORT_JSONL_OUTPUT to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )
    print("JSONL inputs:")
    for p in paths:
        print(" -", p)

    try:
        asyncio.run(repl(paths))
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(repl(paths))
        else:
            raise

if __name__ == "__main__":
    main()


JSONL inputs:
 - metadata_jsonl/classicmodels.jsonl



MCP tools: list_tables, query_db, query_db_full


Enter a test question (or press Enter to skip):  show me customers name



=== RAG Ingest Report ===
Added (first run): 0
Updated (upserted): 0
Skipped (unchanged): 59
Total docs now in collection: 59
Chroma path: /app/chroma_db | Collection: classicmodels

--- Generated Answer (RAG) ---

SELECT DISTINCT c.customerName FROM customers c ;

-- notes: Simple query on customers table.

Available tables:
   1. classicmodels.customers
   2. classicmodels.employees
   3. classicmodels.offices
   4. classicmodels.orderdetails
   5. classicmodels.orders
   6. classicmodels.payments
   7. classicmodels.productlines
   8. classicmodels.products
   9. datahub.metadata_aspect_v2

Total: 9 table(s)

SQL (RAG -> MCP):
SELECT DISTINCT c.customerName FROM customers c;

Query Result (preview):
+-----+----------------------------+
|     |        customerName        |
+-----+----------------------------+
|  1  |     Atelier graphique      |
|  2  |     Signal Gift Stores     |
|  3  | Australian Collectors, Co. |
|  4  |     La Rochelle Gifts      |
|  5  |     Baane Mini Impor


Export FULL result to Excel/CSV? [y/N]:  n



Ask me more questions! (type 'exit' to quit)



Question>  show me customers name



--- Generated Answer (RAG) ---

SELECT DISTINCT c.customerName FROM customers c ;

-- notes: Simple query on customers table.

Available tables:
   1. classicmodels.customers
   2. classicmodels.employees
   3. classicmodels.offices
   4. classicmodels.orderdetails
   5. classicmodels.orders
   6. classicmodels.payments
   7. classicmodels.productlines
   8. classicmodels.products
   9. datahub.metadata_aspect_v2

Total: 9 table(s)

SQL (RAG -> MCP):
SELECT DISTINCT c.customerName FROM customers c;

Query Result (preview):
+-----+----------------------------+
|     |        customerName        |
+-----+----------------------------+
|  1  |     Atelier graphique      |
|  2  |     Signal Gift Stores     |
|  3  | Australian Collectors, Co. |
|  4  |     La Rochelle Gifts      |
|  5  |     Baane Mini Imports     |
| ... |            ...             |
+-----+----------------------------+

Showing preview: 5 rows × 1 columns (full result: 122 rows × 1 columns)
Debug: Query returned 5 row


Export FULL result to Excel/CSV? [y/N]:  n

Question>  show me customer names



--- Generated Answer (RAG) ---

SELECT DISTINCT customers.customerName FROM customers ORDER BY customers.customerNumber;

-- notes: Simple query on customers table.

Available tables:
   1. classicmodels.customers
   2. classicmodels.employees
   3. classicmodels.offices
   4. classicmodels.orderdetails
   5. classicmodels.orders
   6. classicmodels.payments
   7. classicmodels.productlines
   8. classicmodels.products
   9. datahub.metadata_aspect_v2

Total: 9 table(s)

SQL (RAG -> MCP):
SELECT DISTINCT customers.customerName FROM customers ORDER BY customers.customerNumber;
query_db failed: Count failed: (3065, "Expression #1 of ORDER BY clause is not in SELECT list, references column 'classicmodels.customers.customerNumber' which is not in SELECT list; this is incompatible with DISTINCT")
